In [3]:
import pandas as pd
from neutrophil_shape.config.loader import load_config
from neutrophil_shape.CustomFunctions import DetailedBalance, utils

In [7]:
### load config stuff
config = load_config(microscope_type = 'confocal')
#set alignment
config._alignment = 'trajectory_shape'
savedir = config.common.savedir
datadir = savedir / 'shape_data'
dbdir = savedir / 'detailed_balance'
npcs = config.common.npcs
ntrans = config.db_params.ntrans
origins = config.db_params.origins
pc_combos = config.common.pc_combos
sym_pc_combos = config.common.pc_combos_sym_only
all_pc_combos = sym_pc_combos
####### load common directories and data
FullFrame = pd.read_csv(datadir.joinpath('All_Data_with_CGPS_bins.csv'), index_col=0)
centers = pd.read_csv(datadir.joinpath('PC_bin_centers.csv'), index_col=0)

## restrict the treatments and PCs specifically for bootstrapping
bstreats = []#['Random','Galvanotaxis', 'DMSO', 'CK666','Para-Nitro-Blebbistatin']
bspcs = []#[(1,7)]#[(1,2),(4,5),(2,8)]
alldatabs = True

In [3]:
############# create all CGPSs #############
for whichpcs in all_pc_combos:
    if __name__ ==  '__main__':
        #pass if we want to restrict PCs
        if len(bspcs)>0 and whichpcs not in bspcs:
                continue
        ########### get raw transitions and pairs ###########
        rawtrans = DetailedBalance.get_raw_cgps_trajectories(
                FullFrame, #pandas dataframe with all of the cgps binned data
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,
                )

        ########### interpolate all transitions so that only individual transitions are made ###########
        transdf_sep = DetailedBalance.get_interpolated_cgps_trajectories(
                rawtrans, #pandas dataframe with all of the cgps binned data
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,
                )

        ############## get the counts of cells leaving 
        trans_rate_df_sep = DetailedBalance.aggregate_transition_counts(
                transdf_sep, #transdf_sep from get_interpolated_cgps_trajectories
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,
                )

Aggregating PC1-PC2_abs transitions


100%|██████████| 4382/4382 [00:17<00:00, 256.80it/s]


Interpolating PC1-PC2_abs trajectories


100%|██████████| 4375/4375 [00:16<00:00, 271.45it/s]


Aggregating PC1-PC5_abs transitions


100%|██████████| 4382/4382 [00:17<00:00, 247.05it/s]


... (78 outputs left out)

Aggregating PC4-PC6_abs transitions


100%|██████████| 4382/4382 [00:22<00:00, 196.09it/s]


Interpolating PC4-PC6_abs trajectories


100%|██████████| 4366/4366 [00:22<00:00, 193.22it/s]


In [8]:
############# measure aer and cycling frequencies from the observed raw data ###########

for whichpcs in all_pc_combos:
    #pass if we want to restrict PCs
    if len(bspcs)>0 and whichpcs not in bspcs:
        continue

    wpc_str = utils.whichpc_string(whichpcs)
    ### open the raw transitions
    rawtrans = pd.read_csv(dbdir.joinpath(
            f'{wpc_str}_raw_transitions.csv'), index_col=0)

    ############# measure aer and cycling frequencies ###########
    allaers, lrrdf = DetailedBalance.get_aer_cf(
        rawtrans,
        whichpcs, #which two PCs to use in the cgps [x,y]
        config,
        'CellID',
        )
    ### save aer stuff
    allaers.to_csv(dbdir.joinpath(f'{utils.whichpc_string(whichpcs)}_raw_transition_aer_cf.csv'))
    lrrdf.to_csv(dbdir.joinpath(f'{utils.whichpc_string(whichpcs)}_raw_transition_linear_rates.csv'))




100%|██████████| 1776/1776 [00:15<00:00, 111.17it/s]


In [9]:
########### bootstrap transitions and calculate all the aers and cfs around all the pairwise cgps ###############
for whichpcs in all_pc_combos:
    #pass if we want to restrict PCs
    if len(bspcs)>0 and whichpcs not in bspcs:
        continue
    ## use a separate savedir to bootstrap using all data
    bssavestr = 'alldatabs' if alldatabs else 'separatedatabs'
    dbbssavedir = config.common.savedir / 'detailed_balance' / bssavestr
    if __name__ ==  '__main__':
        wpc_str = utils.whichpc_string(whichpcs)
        #### open the transitions
        rawtrans = pd.read_csv(dbdir.joinpath(
            f'{wpc_str}_raw_transitions.csv'), index_col=0)
        interpolated_trans = pd.read_csv(dbdir.joinpath(
            f'{wpc_str}_interpolated_transitions.csv'), index_col=0)
        raw_aer_cf = pd.read_csv(dbdir.joinpath(
            f'{wpc_str}_raw_transition_aer_cf.csv'), index_col=0)

        #merge all treatments if bootstrapping with all data
        if alldatabs:
            rawtrans.loc[:,'Treatment'] = 'alldata'
            interpolated_trans.loc[:,'Treatment'] = 'alldata'
            raw_aer_cf.loc[:,'Treatment'] = 'alldata'
        #restrict to bootstrapped treatments if desired
        if len(bstreats)>0:
            rawtrans = rawtrans[rawtrans.Treatment.isin(bstreats)]
            interpolated_trans = interpolated_trans[interpolated_trans.Treatment.isin(bstreats)]
            raw_aer_cf = raw_aer_cf[raw_aer_cf.Treatment.isin(bstreats)]

        ############## BOOTSTRAP MANY TRAJECTORIES (also computes and saves the
        ############## average bootstrapped currents internally) ##########
        bstrans, bsframe_sep_full = DetailedBalance.get_bootstrapped_cgps_trajectories(
                rawtrans, #raw transition pairs from get_raw_cgps_trajectories
                interpolated_trans, #already-interpolated transitions from get_interpolated_cgps_trajectories
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,
                dbbssavedir, #where to save the bootstrapped dataframes
                )

        if not alldatabs:
            ############# match bootstrapped data to the real dataset's track-length distribution ##########
            bstrans_full = DetailedBalance.construct_bstrans_from_lookup(bstrans, rawtrans)
            replicate_df = DetailedBalance.match_dataset_distribution(rawtrans, bstrans_full)
            replicate_df.to_csv(dbbssavedir.joinpath(f'{utils.whichpc_string(whichpcs)}_real_data-matched_bootstrapped_indices.csv'))

        ############# measure aer and cycling frequencies for all bootstrap iters,
        ############# by lookup against raw_aer_cf 
        allaers = DetailedBalance.construct_bstrans_from_lookup(bstrans, raw_aer_cf)
        lrrdf = DetailedBalance.get_linear_rates(allaers, 'iter')
        lrrdf.to_csv(dbbssavedir.joinpath(f'{utils.whichpc_string(whichpcs)}_bootstrapped_{ntrans}_linear_rates.csv'))
        if not alldatabs:
            ############# measure aer and cycling frequencies for "real distribution" replicate iters ###########
            bs_ind_df = pd.read_csv(dbbssavedir.joinpath(f'{utils.whichpc_string(whichpcs)}_real_data-matched_bootstrapped_indices.csv'),
                                    index_col = 0)
            allaers_repl = allaers.iloc[bs_ind_df.bs_indices.values].reset_index(drop = True)
            lrrdf_repl = DetailedBalance.get_linear_rates(allaers_repl, 'iter')
            lrrdf_repl.to_csv(dbbssavedir.joinpath(f'{utils.whichpc_string(whichpcs)}_replicated_{ntrans}_linear_rates.csv'))

Looking up interpolated trajectories for alldata
Calculating bootstrapped CGPS transition rates for alldata


100%|██████████| 3000/3000 [00:16<00:00, 177.64it/s]


Finished bootstrapping


100%|██████████| 3000/3000 [00:16<00:00, 177.81it/s]


Looking up interpolated trajectories for alldata
Calculating bootstrapped CGPS transition rates for alldata


100%|██████████| 3000/3000 [00:20<00:00, 144.13it/s]


... (78 outputs left out)

Looking up interpolated trajectories for alldata
Calculating bootstrapped CGPS transition rates for alldata


100%|██████████| 3000/3000 [00:22<00:00, 131.17it/s]


Finished bootstrapping


100%|██████████| 3000/3000 [00:17<00:00, 173.91it/s]
